# 00 – Data Quality Audit (M2)
**Project:** E-commerce Delivery & Logistics Performance Analysis
**Owner:** M2 – Data Quality Analyst

**Purpose:** Inspect the raw Olist data for quality problems *before* cleaning, so every later rule is backed by a number.

**Rules followed**
- Raw files in `data/raw/` are only **read**, never modified (SHA-256 fingerprints saved in `docs/raw_file_checksums.txt`).
- Unusual records are **flagged, not deleted**. Every decision is written down with its reason.

In [1]:
# Import the libraries we need
import pandas as pd   # pandas: work with tables (DataFrames)
import numpy as np    # numpy: fast number calculations

RAW = "../data/raw/"  # path to raw data; ".." = go one folder up from notebooks/

In [2]:
# Load the 7 tables needed for Project 1
# Geolocation is skipped: the project has no distance field, and one zip prefix maps to many coordinates
orders      = pd.read_csv(RAW + "olist_orders_dataset.csv")         # one row per order
items       = pd.read_csv(RAW + "olist_order_items_dataset.csv")    # one row per item inside an order
reviews     = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")  # customer ratings (1-5 stars)
products    = pd.read_csv(RAW + "olist_products_dataset.csv")       # product details and category
customers   = pd.read_csv(RAW + "olist_customers_dataset.csv")      # customer city and state
sellers     = pd.read_csv(RAW + "olist_sellers_dataset.csv")        # seller city and state
translation = pd.read_csv(RAW + "product_category_name_translation.csv")  # Portuguese -> English category names

orders.head()  # show the first 5 rows to see what the data looks like

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


**Reading the orders table.** Each order moves through 4 timestamps, always in this order:

① `order_purchase_timestamp` (customer buys) → ② `order_approved_at` (payment approved) → ③ `order_delivered_carrier_date` (handed to courier) → ④ `order_delivered_customer_date` (customer receives).

`order_estimated_delivery_date` is the promised date shown to the customer. Any order where these happen in the wrong sequence is impossible and gets flagged in Check 6.

## Step 2: Structure overview

In [3]:
# CHECK 1: How big is each table, and are there exact duplicate rows?
# Put all tables in one dictionary so a single loop can check them all
tables = {"orders": orders, "items": items, "reviews": reviews,
          "products": products, "customers": customers,
          "sellers": sellers, "translation": translation}

for name, df in tables.items():
    # len(df) = rows, df.shape[1] = columns, df.duplicated() = rows that are a full copy of an earlier row
    print(f"{name:12} rows={len(df):>7}  cols={df.shape[1]:>2}  duplicate_rows={df.duplicated().sum()}")

orders       rows=  99441  cols= 8  duplicate_rows=0
items        rows= 112650  cols= 7  duplicate_rows=0
reviews      rows=  99224  cols= 7  duplicate_rows=0
products     rows=  32951  cols= 9  duplicate_rows=0
customers    rows=  99441  cols= 5  duplicate_rows=0
sellers      rows=   3095  cols= 4  duplicate_rows=0
translation  rows=     71  cols= 2  duplicate_rows=0


**Finding (Check 1):** Row counts match M1's provenance note (orders 99,441 · items 112,650 · reviews 99,224 · products 32,951 · customers 99,441 · sellers 3,095 · translation 71).
**No exact duplicate rows in any of the 7 tables.** No action needed.

In [4]:
# CHECK 2: Missing values per column (only columns that have at least one gap)
for name, df in tables.items():
    missing = df.isna().sum()          # count of empty cells per column
    missing = missing[missing > 0]     # keep only columns with gaps
    if len(missing) > 0:
        print(f"\n== {name} ==")
        print(pd.DataFrame({"missing": missing,
                            "pct": (missing / len(df) * 100).round(2)}))  # same count as % of rows


== orders ==
                               missing   pct
order_approved_at                  160  0.16
order_delivered_carrier_date      1783  1.79
order_delivered_customer_date     2965  2.98

== reviews ==
                        missing    pct
review_comment_title      87656  88.34
review_comment_message    58247  58.70

== products ==
                            missing   pct
product_category_name           610  1.85
product_name_lenght             610  1.85
product_description_lenght      610  1.85
product_photos_qty              610  1.85
product_weight_g                  2  0.01
product_length_cm                 2  0.01
product_height_cm                 2  0.01
product_width_cm                  2  0.01


**Finding (Check 2):** Missing does not always mean error. Each gap was judged on whether it is logical.

| Table / column | Missing | % | Assessment |
|---|---:|---:|---|
| orders · `order_approved_at` | 160 | 0.16 | Investigate by status (Check 7) |
| orders · `order_delivered_carrier_date` | 1,783 | 1.79 | Mostly non-dispatched orders; check delivered ones (Check 7) |
| orders · `order_delivered_customer_date` | 2,965 | 2.98 | Mostly non-delivered orders; check delivered ones (Check 7) |
| reviews · `review_comment_title` | 87,656 | 88.34 | **Normal**: optional free-text field |
| reviews · `review_comment_message` | 58,247 | 58.70 | **Normal**: optional free-text field |
| products · `product_category_name` | 610 | 1.85 | **Keep**, group as "Unknown" in analysis |
| products · name/description length, photos | 610 | 1.85 | Same 610 products; not used in Project 1 |
| products · weight / dimensions | 2 | 0.01 | Not used in Project 1; note only |

Note: source column names `product_name_lenght` and `product_description_lenght` are misspelled in the original Olist data. Left unchanged in raw.

## Step 3: Data quality checks

In [5]:
# CHECK 3: Does any order_id appear more than once in the orders table?
# The orders table must be one row per order, otherwise every KPI would double-count
print("Duplicate order_id:", orders["order_id"].duplicated().sum())

Duplicate order_id: 0


**Finding (Check 3):** 0 duplicate `order_id`. Orders table grain confirmed: **one row per order**.

In [6]:
# CHECK 4: Are status labels consistent? (no 'Delivered' vs 'delivered', no stray spaces)
orders["order_status"].value_counts()  # each distinct status and how many orders have it

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

**Finding (Check 4):** 8 clean, lowercase status values, no spelling variants.

| Status | Orders |
|---|---:|
| delivered | 96,478 |
| shipped | 1,107 |
| canceled | 625 |
| unavailable | 609 |
| invoiced | 314 |
| processing | 301 |
| created | 5 |
| approved | 2 |

No fix needed. Status will drive the on-time and cancellation KPIs (M6).

In [7]:
# CHECK 5: Can every date be read as a real date?
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]

for col in date_cols:
    # An explicit format avoids day/month mix-ups; errors="coerce" turns unreadable values into NaT instead of crashing
    parsed = pd.to_datetime(orders[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    # Parse failures = gaps after parsing minus gaps that were already there
    failed = parsed.isna().sum() - orders[col].isna().sum()
    print(f"{col:32} could not parse: {failed}")
    orders[col] = parsed  # keep the parsed version (in memory only; the raw CSV is untouched)

order_purchase_timestamp         could not parse: 0
order_approved_at                could not parse: 0
order_delivered_carrier_date     could not parse: 0
order_delivered_customer_date    could not parse: 0
order_estimated_delivery_date    could not parse: 0


**Finding (Check 5):** All 5 date columns parse with one consistent format (`YYYY-MM-DD HH:MM:SS`). **0 parse failures.**
Timestamps are kept as stored in the source (Brazil local time); no timezone conversion applied.

In [8]:
# CHECK 6: Impossible order of events (step ① -> ② -> ③ -> ④ in the wrong sequence)
o = orders  # short alias to keep lines readable

checks = {
    "approved_before_purchase":  o["order_approved_at"] < o["order_purchase_timestamp"],
    "carrier_before_purchase":   o["order_delivered_carrier_date"] < o["order_purchase_timestamp"],
    "carrier_before_approved":   o["order_delivered_carrier_date"] < o["order_approved_at"],
    "delivered_before_purchase": o["order_delivered_customer_date"] < o["order_purchase_timestamp"],
    "delivered_before_carrier":  o["order_delivered_customer_date"] < o["order_delivered_carrier_date"],
    "estimated_before_purchase": o["order_estimated_delivery_date"] < o["order_purchase_timestamp"],
}
# Note: comparing with a missing date (NaT) returns False, so missing dates are not counted here (see Check 7)

for name, mask in checks.items():
    # mask.sum() = number of orders with the problem; mask.mean() = share of all orders
    print(f"{name:28} {mask.sum():>5}  ({mask.mean()*100:.2f}%)")

approved_before_purchase         0  (0.00%)
carrier_before_purchase        166  (0.17%)
carrier_before_approved       1359  (1.37%)
delivered_before_purchase        0  (0.00%)
delivered_before_carrier        23  (0.02%)
estimated_before_purchase        0  (0.00%)


**Finding (Check 6):**

| Check | Orders | % | Assessment |
|---|---:|---:|---|
| approved before purchase | 0 | 0.00 | OK |
| **carrier before purchase** | **166** | 0.17 | Impossible → **flag** |
| **carrier before approved** | **1,359** | 1.37 | Suspicious (likely system timing) → **flag**; affects processing-time KPI (M3) |
| delivered before purchase | 0 | 0.00 | OK |
| **delivered before carrier** | **23** | 0.02 | Impossible → **flag** |
| estimated before purchase | 0 | 0.00 | OK |

In [9]:
# CHECK 7: Do order status and dates agree with each other?
delivered = o["order_status"] == "delivered"  # True for delivered orders

# & = "and", ~ = "not", .isna() = empty, .notna() = not empty
print("Delivered but NO customer delivery date:", (delivered & o["order_delivered_customer_date"].isna()).sum())
print("Delivered but NO carrier date:          ", (delivered & o["order_delivered_carrier_date"].isna()).sum())
print("Delivered but NO approval date:         ", (delivered & o["order_approved_at"].isna()).sum())
print("NOT delivered but HAS delivery date:    ", (~delivered & o["order_delivered_customer_date"].notna()).sum())

# Which statuses do those 'not delivered but has a delivery date' orders have?
o.loc[~delivered & o["order_delivered_customer_date"].notna(), "order_status"].value_counts()

Delivered but NO customer delivery date: 8
Delivered but NO carrier date:           2
Delivered but NO approval date:          14
NOT delivered but HAS delivery date:     6


order_status
canceled    6
Name: count, dtype: int64

**Finding (Check 7):**

| Check | Orders | Assessment |
|---|---:|---|
| Delivered, no customer delivery date | **8** | Matches M1's count. Delivery time cannot be measured → **flag** |
| Delivered, no carrier date | **2** | Dispatch record missing → **flag** |
| Delivered, no approval date | **14** | Approval record missing → **flag** |
| Not delivered but has delivery date | **6** (all `canceled`) | Status and date contradict → **flag** |

In [10]:
# CHECK 8: How long did delivered orders take? Are there extreme values?
d = o[delivered].copy()  # only delivered orders; .copy() so the original table stays untouched

# Delivery time in days = customer delivery date - purchase date (86,400 seconds = 1 day)
d["delivery_days"] = (d["order_delivered_customer_date"] - d["order_purchase_timestamp"]).dt.total_seconds() / 86400

print(d["delivery_days"].describe().round(2))  # count, mean, quartiles, min, max in one go

# IQR rule (Session 8): values above Q3 + 1.5 * (Q3 - Q1) are treated as outliers
q1, q3 = d["delivery_days"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print("\nIQR upper limit (days):", round(upper, 2))
print("Orders above limit:", (d["delivery_days"] > upper).sum())

count    96470.00
mean        12.56
std          9.55
min          0.53
25%          6.77
50%         10.22
75%         15.72
max        209.63
Name: delivery_days, dtype: float64

IQR upper limit (days): 29.15
Orders above limit: 4896


**Finding (Check 8):** 96,470 delivered orders with a delivery date. Mean **12.56 days** (matches M1), median **10.22**, min 0.53, max **209.63 days**.
IQR upper limit = **29.15 days** → **4,896 orders** (≈5%) above it.

**Decision:** keep all of them and **flag** as long deliveries. Delivery times are right-skewed, so the IQR rule also catches genuinely late orders, and late orders are the core topic of this project (M4). Report results with and without them.

In [11]:
# CHECK 9: Are ratings valid, and how do reviews link to orders?
print(reviews["review_score"].value_counts().sort_index())  # how many of each star rating, 1 to 5

per_order = reviews.groupby("order_id").size()  # number of reviews per order
print("\nOrders with more than 1 review:", (per_order > 1).sum())
print("Duplicate review_id:", reviews["review_id"].duplicated().sum())  # same review attached to several orders
print("Reviews for orders NOT in orders table:", (~reviews["order_id"].isin(orders["order_id"])).sum())
print("Orders with NO review:", (~orders["order_id"].isin(reviews["order_id"])).sum())

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Orders with more than 1 review: 547
Duplicate review_id: 814
Reviews for orders NOT in orders table: 0
Orders with NO review: 768


**Finding (Check 9):** All scores are valid (1–5 only): 1★ 11,424 · 2★ 3,151 · 3★ 8,179 · 4★ 19,142 · 5★ 57,328.

| Check | Count | Assessment |
|---|---:|---|
| Orders with >1 review | **547** | **M3 needs a rule** (e.g. use the latest review) |
| Duplicate `review_id` | **814** | One review attached to several orders bought together |
| Reviews with no matching order | 0 | OK |
| Orders with no review | **768** | Excluded from rating comparison only |

In [12]:
print("Products with NO category:", products["product_category_name"].isna().sum())
# CHECK 10: Product categories – missing values and missing English translations
print("Products with NO category:", products["product_category_name"].isna().sum())

# set(A) - set(B) = categories used in products but absent from the translation table
missing_tr = set(products["product_category_name"].dropna()) - set(translation["product_category_name"])
print("Categories with NO English translation:", missing_tr)
print("Products in those categories:", products["product_category_name"].isin(missing_tr).sum())
missing_tr = set(products["product_category_name"].dropna()) - set(translation["product_category_name"])
print("Categories with NO English translation:", missing_tr)
print("Products in those categories:", products["product_category_name"].isin(missing_tr).sum())

Products with NO category: 610
Products with NO category: 610
Categories with NO English translation: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}
Products in those categories: 13
Categories with NO English translation: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}
Products in those categories: 13


**Finding (Check 10):** **610 products** have no category → group as **"Unknown"** (never dropped).
**2 categories** (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) covering **13 products** have no English translation → keep the Portuguese name.

In [13]:
# CHECK 11: Prices and freight – any impossible values?
print("price <= 0:        ", (items["price"] <= 0).sum())          # a free or negative product price is impossible
print("freight_value < 0: ", (items["freight_value"] < 0).sum())   # negative shipping fee is impossible
print("freight_value == 0:", (items["freight_value"] == 0).sum())  # zero freight is possible (free shipping)
items[["price", "freight_value"]].describe().round(2)

price <= 0:         0
freight_value < 0:  0
freight_value == 0: 383


,price,freight_value
count,112650.00,112650.00
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,6735.00,409.68


**Finding (Check 11):** No zero or negative prices (min 0.85, max 6,735.00) and no negative freight.
**383 items** have freight = 0 → treated as free shipping, **kept**. High prices are plausible and kept.

In [14]:
# CHECK 12: Do the tables link correctly, and are region codes clean?
print("Orders with customer_id NOT in customers:", (~orders["customer_id"].isin(customers["customer_id"])).sum())
print("Items with order_id NOT in orders:       ", (~items["order_id"].isin(orders["order_id"])).sum())
print("Orders with NO items:                    ", (~orders["order_id"].isin(items["order_id"])).sum())
print("Customer states:", customers["customer_state"].nunique(), sorted(customers["customer_state"].unique()))

Orders with customer_id NOT in customers: 0
Items with order_id NOT in orders:        0
Orders with NO items:                     775
Customer states: 27 ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']


**Finding (Check 12):** No orphan records (every order has a customer, every item has an order).
**775 orders have no items** → no price/freight for them → **flag**.
Customer states: **27 clean 2-letter codes** (26 Brazilian states + DF). Ready for regional comparison (M4).

## Summary: candidate rules for the audit (Step 4)

| Proposed rule | Orders affected | Action |
|---|---:|---|
| Delivered but no customer delivery date | 8 | Flag |
| Canceled but has customer delivery date | 6 | Flag |
| Delivered but no carrier date | 2 | Flag |
| Delivered but no approval date | 14 | Flag |
| Carrier date before purchase | 166 | Flag |
| Carrier date before approval | 1,359 | Flag |
| Customer delivery before carrier date | 23 | Flag |
| Delivery time above IQR limit (29.15 days) | 4,896 | Flag (keep) |
| Order has no items | 775 | Flag |
| Product has no category | 610 products | Group as "Unknown" |
| Order has more than one review | 547 | Rule needed from M3 |

**No rows are deleted.** Every rule adds a True/False flag so KPI owners (M3, M6) can decide what to exclude for each KPI.